## Import Packages


In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns',100)

## Loading data

In [3]:
train=pd.read_pickle('../data/train_engineered.pkl')


In [4]:
test=pd.read_pickle('../data/train_engineered.pkl')

In [5]:
train.shape

(590540, 453)

In [6]:
test.shape

(590540, 453)

## Baseline Features for Analysis 

In [7]:
baseline_features = [
    "TransactionAmt",
    "TransactionAmt_log",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "dist1",
    "dist2",
    "P_emaildomain_group",
    "R_emaildomain_group",
    "P_emaildomain_suffix",
    "R_emaildomain_suffix",
    "match_email",
    "browser_group",
    "os_group",
    "DeviceType",
    "C1",
    "C2",
    "C13",
    "D1",
    "D10",
    "D15",
    "TransactionAmt_to_mean_card1",
    "TransactionAmt_to_std_card1",
    "TransactionAmt_to_mean_card4",
    "TransactionAmt_to_std_card4"
]

## Dependent & Independent Variables

In [8]:
X = train[baseline_features].copy()
y = train['isFraud'].copy()

## Encode categorical columns

In [9]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = X[col].astype(str)
    X[col] = le.fit_transform(X[col])

## Impute missing values

In [12]:
imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(imputer.fit_transform(X),columns=X.columns)

## Train-test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(472432, 31)
(118108, 31)


## Evaluation Function 

In [14]:
model_results = []

def evaluate_model(model_name, model, X_test_data, y_test):
    
    y_pred = model.predict(X_test_data)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_data)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Model: {model_name}")
    print("=" * 40)
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print(f"ROC AUC: {roc_auc}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    model_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    })